In [1]:
#imports
import numpy as np
from torchvision import datasets as tv_datasets
from torchvision import transforms
import torch
import torch.nn as nn
import gc
import torch.nn.functional as F
from scipy.optimize import minimize_scalar

In [2]:
#model definition
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LNET(nn.Module):
    def __init__(self):
        super(LNET, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(120, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 120)  # Flatten the tensor
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x

In [3]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x)
])

mnist = tv_datasets.MNIST(root='data', train=True, transform=transform, download=True)
mnist_loader = DataLoader(mnist, batch_size=1000, shuffle=True)
print("Loaded MNIST")

cifar = tv_datasets.CIFAR10(root='data', train=True, transform=transform, download=True)
cifar_loader = DataLoader(cifar, batch_size=1000, shuffle=True)
print("Loaded CIFAR-10")

Loaded MNIST
Loaded CIFAR-10


In [4]:
def get_gradient_vector(model, dataloader, device):
    model.to(device)
    model.train()
    x, y = next(iter(dataloader)) #1 batch
    x, y = x.to(device), y.to(device)
    
    model.zero_grad()        
    out = model(x)          
    loss = F.cross_entropy(out, y)   
    loss.backward()                  

    grad_vector = torch.cat([
        p.grad.view(-1) for p in model.parameters() if p.grad is not None
    ])
    
    return grad_vector.detach().cpu().numpy()


In [5]:
def compute_optimal_a_prime_single(g_jt, g_hat_jt):
    if g_hat_jt == 0:
        raise ValueError("g_hat_jt cannot be zero (division by zero)")
    
    def inner_max_expression(a_prime):
        if a_prime >= 1 or a_prime < 0:
            return float('inf')

        ratio = (g_jt / g_hat_jt) ** 2
        term1 = ratio - 1/(1-a_prime)
        term2 = (1-a_prime) - ratio

        return max(term1, term2)
    result = minimize_scalar(inner_max_expression, bounds=(0, 0.999999999), method='bounded')
    return result.x


def compute_optimal_a_prime_vector(g_jt: np.ndarray, g_hat_jt: np.ndarray):
    if np.any(g_hat_jt == 0):
        raise ValueError("g_hat_jt contains zero(s), which would cause division by zero")
    if g_jt.shape != g_hat_jt.shape:
        raise ValueError("g_jt and g_hat_jt must have the same shape")

    def inner_max_expression(a_prime):
        if a_prime >= 1 or a_prime < 0:
            return float('inf')

        # Compute elementwise ratio squared
        ratio = (g_jt / g_hat_jt) ** 2

        # Compute violations per coordinate
        term1 = ratio - 1 / (1 - a_prime)    # violation of right inequality rearranged
        term2 = (1 - a_prime) - ratio        # violation of left inequality rearranged

        # For each coordinate, the violation is max(term1, term2)
        violations = np.maximum(term1, term2)

        # We want the worst violation over all coordinates
        max_violation = np.max(violations)

        return max_violation

    result = minimize_scalar(inner_max_expression, bounds=(0, 0.999999999), method='bounded')
    return result.x

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cifar_model = LNET()
mnist_model = LNET()

cifar_gradients = get_gradient_vector(cifar_model, cifar_loader, device)
mnist_gradients = get_gradient_vector(mnist_model, mnist_loader, device)
print(cifar_gradients.shape)
print(mnist_gradients.shape)

# result = compute_optimal_a_prime_single(cifar_gradients, mnist_gradients)
result = compute_optimal_a_prime_vector(cifar_gradients, mnist_gradients)
print(result)


max_a_prime = 0
for i in range(len(cifar_gradients)):
    g_val = cifar_gradients[i]
    g_hat_val = mnist_gradients[i]
    a_prime_single = compute_optimal_a_prime_single(g_val, g_hat_val)
    if a_prime_single > max_a_prime:
        max_a_prime = a_prime_single

print("Max a' over all coordinates (single scalar opt):", max_a_prime)

(62006,)
(62006,)
0.9999940381390193
Max a' over all coordinates (single scalar opt): 0.9999966493296273
